# Morales2018

https://doi.org/10.1111/pce.13119

This notebook demonstrates example analyses using the Morales et al. (2018) dynamic photosynthesis model implemented in `mxlmodels`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from mxlpy import Simulator, make_protocol
from mxlmodels import get_morales2018

## Load model

In [ ]:
model = get_morales2018()
model

## Irradiance step response

The model is simulated during a rapid increase in irradiance from low light to high light.

In [ ]:
model = get_morales2018()

sim = Simulator(model)
sim.simulate_protocol(
    protocol=make_protocol([
        (300, {"PAR": 50}),
        (3600, {"PAR": 1000}),
    ]),
    time_points_per_step=500,
)

res = sim.get_result().unwrap_or_err().get_combined()
res.head()

## Figure 5-style analysis: NPQ components

Morales et al. (2018) decomposed non-photochemical quenching into additive components associated with energy-dependent quenching (`qE`), chloroplast movement (`qM`), and photoinhibition (`qI`).

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(8, 6))

variables = ["NPQ", "qE", "qM", "qI"]

for ax, var in zip(axs.flat, variables):
    if var in res.columns:
        ax.plot(res.index / 60, res[var])
    elif f"obs_{var}" in res.columns:
        ax.plot(res.index / 60, res[f"obs_{var}"])

    ax.set_title(var)
    ax.set_xlabel("Time / min")
    ax.set_ylabel(var)

plt.tight_layout()
plt.show()

## Photosynthetic outputs

Selected photosynthetic outputs are plotted to inspect the dynamic response of CO2 assimilation, PSII efficiency, stomatal conductance, and Rubisco-limited rates.

In [ ]:
selected_outputs = ["A", "PhiII", "gsw", "Vc", "Vr", "VrJ"]

plt.figure(figsize=(8, 5))

for var in selected_outputs:
    if var in res.columns:
        plt.plot(res.index / 60, res[var], label=var)
    elif f"obs_{var}" in res.columns:
        plt.plot(res.index / 60, res[f"obs_{var}"], label=var)

plt.xlabel("Time / min")
plt.ylabel("Model output")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()